In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("Incremental Flag","0")
increm_flag=dbutils.widgets.get("Incremental Flag")
print(increm_flag)

In [0]:
df_claims_src=spark.sql('''
                     select * from  parquet.`abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/claims_claim_line`''')

In [0]:
df_claims_src.limit(5).display()

In [0]:
df_dim_patient=spark.sql('SELECT * FROM healthcare.gold.dim_patient')
df_dim_payer=spark.sql('SELECT * FROM healthcare.gold.dim_payer')
df_dim_provider=spark.sql('SELECT * FROM healthcare.gold.dim_provider')
df_dim_procedure=spark.sql('SELECT * FROM healthcare.gold.dim_procedure')
df_fact_encounter=spark.sql('SELECT * FROM healthcare.gold.fact_encounter')
df_dim_date=spark.sql('SELECT * FROM healthcare.gold.dim_date')

In [0]:
df_dim_patient.count()

In [0]:
df_dim_payer.count()

In [0]:
df_dim_provider.count()


In [0]:
df_dim_procedure.count()

In [0]:
df_fact_encounter.count()


In [0]:
df_dim_date.count()

In [0]:
df_claims_src.count()

In [0]:
df_staged_fact=df_claims_src.join(df_dim_patient,df_claims_src["mrn"]==df_dim_patient["mrn"],"inner")\
    .join(df_dim_provider,df_claims_src["billing_npi"]==df_dim_provider["npi"],"inner")\
        .join(df_dim_payer,df_claims_src["payer_id"]==df_dim_payer["payer_id"],"inner")\
            .join(df_dim_procedure,df_claims_src["procedure_code"]==df_dim_procedure["procedure_code"],"inner")\
                .join(df_fact_encounter,df_claims_src["encounter_id"]==df_fact_encounter["encounter_id"],"inner")\
                    .join(df_dim_date,df_claims_src["service_date"]==df_dim_date["full_date"],"inner")\
                        .select(df_dim_patient["dim_patient_key"],df_dim_provider["dim_provider_key"],df_dim_payer["dim_payer_key"],df_dim_procedure["dim_procedure_key"],df_fact_encounter["encounter_key"],df_dim_date["dim_date_key"],df_claims_src["billed_amount"],df_claims_src["allowed_amount"],df_claims_src["paid_amount"],df_claims_src["patient_responsibility_amount"],df_claims_src["denial_reason_code"],df_claims_src["claim_status"],df_claims_src["claim_id"])


In [0]:
df_staged_fact.count()

In [0]:
df_staged_fact.limit(5).display()

In [0]:
if increm_flag=='0':
    max_value=1
else:
    max_value_df=spark.sql('''
            select coalesce(max(claim_key),0) from healthcare.gold.fact_claims''')
    max_value=max_value_df.collect()[0][0]+1

In [0]:
if spark.catalog.tableExists('healthcare.gold.fact_claims'):
    df_sink=spark.sql('''
        select claim_key,dim_patient_key,
dim_provider_key,
dim_payer_key,
dim_procedure_key,
encounter_key,
dim_date_key,
billed_amount,
allowed_amount,
paid_amount,
patient_responsibility_amount,
denial_reason_code,
claim_status,
claim_id from healthcare.gold.fact_claims''')
else:
    df_sink=spark.sql('''
            select 1 as claim_key,
billed_amount,
allowed_amount,
paid_amount,
patient_responsibility_amount,
denial_reason_code,
claim_status,
claim_id from parquet.`abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/claims_claim_line` where 1=0''')

In [0]:
df_sink.limit(5).display()

In [0]:
df_sink.count()

In [0]:
df_filter=df_staged_fact.join(df_sink,df_staged_fact["claim_id"]==df_sink["claim_id"],"left")\
    .select(df_staged_fact["*"],df_sink["claim_key"])

In [0]:
df_filter.limit(5).display()


In [0]:
df_new_rec=df_filter.filter(df_filter["claim_key"].isNull())
df_existing_rec=df_filter.filter(df_filter["claim_key"].isNotNull())
df_new_rec.limit(5).display()
df_existing_rec.limit(5).display()

In [0]:
windowSpec = Window.orderBy("claim_id")
df_insert = df_new_rec.withColumn(
    "claim_key", row_number().over(windowSpec) + lit(max_value - 1)
)

In [0]:
df_insert.limit(5).display()

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("healthcare.gold.fact_claims"):
    #insert the new entries
    df_insert.write.format("delta").mode("append").saveAsTable("healthcare.gold.fact_claims")
    #update the entries - deduplicate by claim_key first
    df_existing_rec_dedup = df_existing_rec.dropDuplicates(["claim_key"])
    deltaTable=DeltaTable.forName(spark,"healthcare.gold.fact_claims")
    deltaTable.alias("target").merge(df_existing_rec_dedup.alias("source"),"target.claim_key = source.claim_key")\
        .whenMatchedUpdate(set={
            "dim_patient_key": "source.dim_patient_key",
            "dim_provider_key": "source.dim_provider_key",
            "dim_procedure_key": "source.dim_procedure_key",
            "dim_payer_key": "source.dim_payer_key",
            "dim_date_key": "source.dim_date_key",
            "encounter_key": "source.encounter_key",
            "billed_amount": "source.billed_amount",
            "allowed_amount": "source.allowed_amount",
            "paid_amount": "source.paid_amount",
            "patient_responsibility_amount": "source.patient_responsibility_amount",
            "denial_reason_code": "source.denial_reason_code",
            "claim_status": "source.claim_status",
            "claim_id": "source.claim_id"
    }).execute()
else:
    df_insert.write.format("delta").mode("append").saveAsTable("healthcare.gold.fact_claims")

In [0]:
%sql
select * from healthcare.gold.fact_claims